# K/V pilot — chuẩn bị dữ liệu trên CPU
**Accelerator: None · Internet ON · Save & Run All. Không cần GPU.**

Notebook chỉ lọc official TRAIN/nhãn được quan sát, ghép GT với predicted boxes,
và sao chép CLIP/legacy tensors có sẵn. Không dựng hoặc chạy captioner/encoder;
checkpoint C0 chỉ được kiểm SHA. Không training/SCST/đánh giá caption ở đây.

Add Input giống lượt expanded: `concept-mass-c-c0-v111-v112`, `prompt-cache`,
`visual-cache`, `objectdetectionecache`, `mscoco-2014` của vuthetam.
Kiểm các đường dẫn ở cell đầu. Repo nằm trong /tmp; Output không chứa toàn source repo.

Metadata COCO chính thức được tải thêm (~152 MiB compressed của hai member, không ảnh).
Dữ liệu có thể chưa đủ sau GT/top6/ontology checks: notebook sẽ dừng ở trạng thái
NEEDS_SOURCE_REVIEW thay vì tự nới tiêu chuẩn. Đây không phải test CUDA/model.
Không bật GPU để chạy bước này. Giới hạn7200s là cap, không ETA.


In [ ]:
from pathlib import Path
import tempfile, sys, json, time, hashlib, urllib.request, importlib.util
import subprocess, threading, queue, os, signal, zipfile
print('K/V CPU preparation starting — no model, no training.', flush=True)
CHECKPOINT=Path('/kaggle/input/datasets/ducanh2403/concept-mass-c-c0-v111-v112/C0.bin')
COCO_JSON=Path('/kaggle/input/datasets/vuthetam/mscoco-2014/dataset_coco.json')
LEGACY=Path('/kaggle/input/datasets/ducanh2403/prompt-cache/prompt_clip_tokens_cache.pt')
VISUAL=Path('/kaggle/input/datasets/ducanh2403/visual-cache')
DETECTIONS=Path('/kaggle/input/datasets/ducanh2403/objectdetectionecache/objectdetectioncache.json')
OUTPUT=Path('/kaggle/working/relation_kv_preparation_v1')
LOG=OUTPUT.with_suffix('.log')
REVIEW=OUTPUT.with_name(OUTPUT.name+'_review.zip')
for p in (CHECKPOINT,COCO_JSON,LEGACY,DETECTIONS):
    if not p.is_file():raise FileNotFoundError(f'Edit the input path in this cell: {p}')
if not VISUAL.is_dir():raise FileNotFoundError(VISUAL)
if OUTPUT.exists() or LOG.exists() or REVIEW.exists():
    raise FileExistsError('Preserve existing output; use a fresh session or edit OUTPUT.')
WORK=Path(tempfile.mkdtemp(prefix='relation-kv-cpu-'))
REPO=WORK/'repo'; REPO.mkdir()
PREP_SECONDS=7200
print('Inputs found. Select Accelerator=None for this notebook.', flush=True)


In [ ]:
CORE='bfc54da3078ce241282643a4018698942f67f253'
ASSETS={'experiments/semantic_grounding_method/kv_data.py': '2d6fec1d00fea6dbc34396b60f868549ac86d17b6410f56ff47e3cf01ba28e39', 'experiments/semantic_grounding_method/prepare_kv_cpu.py': '9d4f6db2d9483d44f1f891520b2eb5fd8e186d84edfc86018b7945171eecde6b', 'experiments/semantic_grounding_method/test_kv_data.py': 'd0ad46c4b7df313ffdbc56f9a2a0495795711c7a69c2e26f3388419e0b61ab4d', 'experiments/semantic_grounding_method/test_relation_kv_math.py': '0b3eca5419d20e6a544b1b22372b37d0e88e30a5d02e7f3737d7072f99cd2014', 'experiments/semantic_grounding_method/test_kv_source_fetch.py': '7e8aefe73db31e0a71b070ed809b5242f48d4864183c69ab8a0bd144f073e3ff', 'experiments/semantic_grounding_method/data/gain_metadata.json.gz': '62f43f771aec9feba31a2fbbb6fe7db2878f259979faac378b74a754822c7b67', 'experiments/relation_frames_inventory/inventory.py': '8975c6a221e3d769c5449901347717f8776c4ae191fb4fad5d3eb117fe18e578', 'experiments/relation_frames_inventory/sources/source_receipt.json': 'c54bbd21e7e77b0a64c87bcfe67d98f875b3f53ba75e782221c921ac0c47d05d', 'experiments/relation_frames_inventory/V2_MANIFEST.json.gz': '6ca6af23d0a31043fecdf33398f5b60098be07cbeea9fb91cbfec4b314c2cfa7', 'experiments/relation_frame_pretrain/frame_data.py': '87c58a50142f68b1721323f7c53fa08e256bb5bc8aee3d7625262161ae1692de', 'experiments/concept_mass_grounding/TRAIN_PRESENCE.json.gz': 'dd888949a1a8cc363efd760eb46012a7a6f736ac6b167d98be5303393b614117', 'fetch_vcoco_coco_instances.py': '790588f4cde7310f6153900d8aa5a6fb0a68856074c4992f90a6250481fcf4ea', 'audit_vcoco_alignment_data.py': '610453bc0ba56297b7b1966bf2ff2cb8926010d37a56c062d0dbceffc79f904e', 'captioning/__init__.py': 'ad259f50a32265d298d027a888f801d378c4669313b8dab06e0653978ce30633', 'captioning/object_consistency.py': '5fbd75b0ff4a35f0475d460a294894009d6007f468e248a0e4b528aaa8fabf73', 'captioning/supervised_relations.py': '3e29fd60bbf166ae078581fb9ad34da6ce4462ee287f1c4eb9bbda2abd91f984', 'captioning/semantic_prompts.py': 'a06f12dea63604f9db9d19b6922815db27a35b648a17059407ffaf26a2a0d511', 'captioning/visual_cache.py': 'adbcc9678fff499fc767cdbf04a4966186a40729f51c5bfb2525e79f9b6b5fe1'}
for asset_index, (relative, expected) in enumerate(ASSETS.items(), 1):
    url='https://raw.githubusercontent.com/Supzxjee/Image_Captioning/'+CORE+'/'+relative
    with urllib.request.urlopen(url,timeout=60) as response:raw=response.read()
    if hashlib.sha256(raw).hexdigest()!=expected:raise ValueError('Changed source: '+relative)
    target=REPO/relative;target.parent.mkdir(parents=True,exist_ok=True);target.write_bytes(raw)
    if asset_index%5==0 or asset_index==len(ASSETS):print('Verified source files:',asset_index,'/',len(ASSETS),flush=True)
for package in ('numpy','h5py','torch'):
    if importlib.util.find_spec(package) is None:raise RuntimeError('Kaggle runtime lacks '+package)
if importlib.util.find_spec('ijson') is None:
    subprocess.run([sys.executable,'-m','pip','install','--quiet','ijson==3.4.0'],check=True)
HERE=REPO/'experiments/semantic_grounding_method/'
CONFIG=WORK/'config.json'
cfg=dict(checkpoint=str(CHECKPOINT),dataset_json=str(COCO_JSON),legacy_prompt=str(LEGACY),
         visual_cache=str(VISUAL),detections=str(DETECTIONS),output=str(OUTPUT),
         work=str(WORK),source_commit=CORE)
CONFIG.write_text(json.dumps(cfg,indent=2),encoding='utf-8')
print('Verified pinned source:',CORE,'CPU preparation only.',flush=True)


In [ ]:
def run(args, timeout):
    print('Running:', ' '.join(map(str,args)),flush=True)
    process=subprocess.Popen(list(map(str,args)),cwd=REPO,stdout=subprocess.PIPE,
                             stderr=subprocess.STDOUT,text=True,bufsize=1,start_new_session=True)
    messages=queue.Queue()
    def reader():
        try:
            for line in process.stdout:messages.put(line)
        finally:messages.put(None)
    thread=threading.Thread(target=reader,daemon=True);thread.start()
    start=heartbeat=time.monotonic();ended=False
    with LOG.open('a',encoding='utf-8') as log:
        try:
            while not ended or process.poll() is None or not messages.empty():
                try:
                    line=messages.get(timeout=1)
                    if line is None:ended=True
                    else:print(line,end='',flush=True);log.write(line);log.flush()
                except queue.Empty:pass
                now=time.monotonic()
                if now-heartbeat>=30:
                    print(f'CPU worker active; {now-start:.0f}s elapsed. No model/training.',flush=True)
                    heartbeat=now
                if now-start>timeout:raise TimeoutError('CPU phase exceeded locked cap')
            code=process.wait()
            if code:raise subprocess.CalledProcessError(code,args)
        finally:
            if process.poll() is None:
                os.killpg(process.pid,signal.SIGTERM)
                try:process.wait(timeout=10)
                except subprocess.TimeoutExpired:os.killpg(process.pid,signal.SIGKILL);process.wait()
            thread.join(timeout=5)

try:
    for test in ('test_kv_data.py','test_relation_kv_math.py','test_kv_source_fetch.py'):
        run([sys.executable,'-u',HERE/test],120)
    run([sys.executable,'-u',HERE/'prepare_kv_cpu.py','--config',CONFIG],PREP_SECONDS)
finally:
    with zipfile.ZipFile(REVIEW,'x',compression=zipfile.ZIP_DEFLATED) as archive:
        for p in sorted(OUTPUT.glob('*.json')):
            archive.write(p,p.name)
        if LOG.exists():archive.write(LOG,'notebook.log')
        archive.write(CONFIG,'requested_config.json')
        for name in ('kv_data.py','prepare_kv_cpu.py'):
            archive.write(HERE/name,'source/'+name)
    print('SEND REVIEW ZIP:',REVIEW,flush=True)
    print('KEEP ENTIRE FOLDER:',OUTPUT,flush=True)
receipt=OUTPUT/'cpu_preparation.json'
status=json.loads(receipt.read_text()).get('status') if receipt.is_file() else 'INCOMPLETE'
print('Status:',status,'— model/CUDA have NOT run.',flush=True)
print('Send the review ZIP before starting D/E training.',flush=True)
